In [1]:
from visualise import visualise, load_model

In [2]:
import os
root_path = os.path.abspath("./predictions")
print(root_path)

c:\Users\mokrota\Documents\GitHub\Neural-Network-Project\project\predictions


In [3]:
from torch.utils.data import DataLoader, Subset
from torchvision.datasets import EMNIST
import torchvision.transforms as transforms

to_tensor = transforms.Compose([
    transforms.Grayscale(num_output_channels=1), transforms.ToTensor()
])
dataset = EMNIST(root="data_emnist", split="letters", train=False, transform=to_tensor)

In [4]:
model_path = os.path.abspath("./best_models/best_pyramid/cnn20.pth")
metadata_path = os.path.abspath("./best_models/best_pyramid/train_metadata20.json")
r_model_path = os.path.abspath("./best_models/best_reverse_pyramid/cnn10.pth")
r_metadata_path = os.path.abspath("./best_models/best_reverse_pyramid/train_metadata10.json")

In [5]:
save_path_root = os.path.abspath("./predictions/emnist_analysis/")
if not os.path.exists(save_path_root):
    os.mkdir(save_path_root)

In [6]:
def filter_dataset(dataset, label: int):
    filtered_i = [i for i, label in enumerate(dataset.targets) if label in int_combo]
    filtered_dataset = Subset(dataset, filtered_i)
    return filtered_dataset

In [7]:
import torch
def filter_misclassified_samples(pred, gt, condition):
    # condition is function that takes (predicted_label, true_label) and return bool
    filtered_l = []
    for i, (p, g) in enumerate(zip(pred, gt)):
        if condition(p, g):
            filtered_l.append(i)
    return filtered_l

In [8]:
def get_label(l):
    return ord(l) - ord('a') + 1

In [9]:
model = load_model(model_path, metadata_path)
device = torch.device('cuda')
model.to(device)
model.eval()
pred = []
gt = []

dataloader = DataLoader(dataset, batch_size=32, shuffle=False)

with torch.no_grad():
    for inputs, labels in dataloader:
        inputs, labels = inputs.to(device), labels.to(device)
        outputs = model(inputs)
        predictions = torch.argmax(outputs, dim=1) + 1
        for pred_label, true_label in zip(predictions, labels):
            pred.append(pred_label.item())
            gt.append(true_label.item())

c:\Users\mokrota\Documents\GitHub\Neural-Network-Project\.venv\Lib\site-packages\torch\nn\modules\conv.py:549: UserWarning: Using padding='same' with even kernel lengths and odd dilation may require a zero-padded copy of the input be created (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\pytorch\aten\src\ATen\native\Convolution.cpp:1037.)
  return F.conv2d(
c:\Users\mokrota\Documents\GitHub\Neural-Network-Project\.venv\Lib\site-packages\torch\nn\modules\module.py:1739: UserWarning: Implicit dimension choice for softmax has been deprecated. Change the call to include dim=X as an argument.
  return self._call_impl(*args, **kwargs)


In [10]:
def visualise_combo(model, true_label: str, predicted_label: str, pred, gt, dataset, path, img_num=50):
    filter_idxs = filter_misclassified_samples(pred, gt, lambda p, t: p == get_label(predicted_label) and t == get_label(true_label))
    filter_dataset = Subset(dataset, filter_idxs)
    dataloader = DataLoader(filter_dataset, batch_size=img_num, shuffle=False)
    return visualise(model, dataloader=dataloader, path=path)

In [11]:
true_labels = ["d", "d",
               "g", "g",
               "i", "i",
               "l", "l"]
predicted_labels = ["d", "o", 
                    "g", "q",
                    "i", "l",
                    "l", "i"]

paths = [os.path.join(save_path_root, f"p-{p}_t-{t}") for p, t in zip(predicted_labels, true_labels)]

In [12]:
for t_label, p_label, path in zip(true_labels, predicted_labels, paths):
    print(path)
    visualise_combo(model, t_label, p_label, pred, gt, dataset, path,img_num=4)

c:\Users\mokrota\Documents\GitHub\Neural-Network-Project\project\predictions\emnist_analysis\p-d_t-d


FileExistsError: [WinError 183] Cannot create a file when that file already exists: 'c:\\Users\\mokrota\\Documents\\GitHub\\Neural-Network-Project\\project\\predictions\\emnist_analysis\\p-d_t-d\\image_0'

In [ ]:
reverse_save_path_root = os.path.abspath("./predictions/emnist_analysis_reverse")
if not os.path.exists(reverse_save_path_root):
    os.mkdir(reverse_save_path_root)

In [ ]:
r_model = load_model(r_model_path, r_metadata_path)
device = torch.device('cuda')
r_model.to(device)
r_model.eval()
r_pred = []
r_gt = []

dataloader = DataLoader(dataset, batch_size=32, shuffle=False)

with torch.no_grad():
    for inputs, labels in dataloader:
        inputs, labels = inputs.to(device), labels.to(device)
        outputs = r_model(inputs)
        predictions = torch.argmax(outputs, dim=1) + 1
        for pred_label, true_label in zip(predictions, labels):
            r_pred.append(pred_label.item())
            r_gt.append(true_label.item())

c:\Users\mokrota\Documents\GitHub\Neural-Network-Project\.venv\Lib\site-packages\torch\nn\modules\module.py:1739: UserWarning: Implicit dimension choice for softmax has been deprecated. Change the call to include dim=X as an argument.
  return self._call_impl(*args, **kwargs)


In [ ]:
r_true_labels = ["l",
               "t", "t", "t"]
r_predicted_labels = ["i", 
                    "f", "j", "r"]

r_paths = [os.path.join(reverse_save_path_root, f"p-{p}_t-{t}") for p, t in zip(r_predicted_labels, r_true_labels)]

In [ ]:
for t_label, p_label, path in zip(r_true_labels, r_predicted_labels, r_paths):
    print(path)
    visualise_combo(r_model, t_label, p_label, r_pred, r_gt, dataset, path, img_num=4)

c:\Users\mokrota\Documents\GitHub\Neural-Network-Project\project\predictions\emnist_analysis_reverse\p-i_t-l
c:\Users\mokrota\Documents\GitHub\Neural-Network-Project\project\predictions\emnist_analysis_reverse\p-f_t-t
c:\Users\mokrota\Documents\GitHub\Neural-Network-Project\project\predictions\emnist_analysis_reverse\p-j_t-t
c:\Users\mokrota\Documents\GitHub\Neural-Network-Project\project\predictions\emnist_analysis_reverse\p-r_t-t
